# 01 — Data, tokenizer and batches

Before any neural network we need to (1) look at the raw text, (2) turn characters
into numbers, (3) split the data and (4) cut it into mini-batches of inputs/targets.

## Experimenting with the text file

In [1]:
with open('../data/wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print('characters:', len(text))
print(text[:500])

characters: 232307
DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, OZMA OF OZ, ETC.

  ILLUSTRATED BY JOHN R. NEILL

  BOOKS OF WONDER WILLIAM MORROW & CO., INC. NEW YORK


  [Illustration]


  COPYRIGHT 1908 BY L. FRANK BAUM

  ALL RIGHTS RESERVED


         *       *       *       *       *


  [Illustration]


  DEDICATED TO HARRIET A. B. NEAL.


         *       *       *       *       *


To My Readers


It's no use; no use at all. The children won't let me s


Every distinct character becomes one entry in our **vocabulary**.

In [2]:
chars = sorted(set(text))
vocab_size = len(chars)
print(repr(''.join(chars)))
print('vocab size:', vocab_size)

'\n !"&\'()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz'
vocab size: 80


## Character-level tokenizer

A tokenizer maps text ⇄ integers. The simplest one: one token per character, with two
lookup tables — `string_to_int` (encoder) and `int_to_string` (decoder).

In [3]:
string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

encoded_hello = encode('hello')
print(encoded_hello, '->', decode(encoded_hello))

[61, 58, 65, 65, 68] -> hello


## Types of tokenizers

| tokenizer | vocab size | sequence length | example |
|---|---|---|---|
| character | tiny (~80) | very long | `h e l l o` |
| word | huge (100k+) | short | `hello` |
| subword (BPE / WordPiece / SentencePiece) | medium (32k–100k) | medium | `hel lo` |

Character-level keeps the vocabulary tiny (small embedding table, no unknown words) at
the cost of longer sequences. GPT-2/3/4 use byte-level BPE (`tiktoken`). See
`docs/01_tokenizers.md` for more.

## Tensors instead of arrays

PyTorch **tensors** are n-dimensional arrays that can live on the GPU and track
gradients. We store the whole encoded corpus as one long 1-D `int64` tensor.

In [4]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([232307]) torch.int64
tensor([28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1, 47, 33,
        50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,  0,  0,
         1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,  0,  1,
         1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47, 33, 50,
        25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36, 25, 38,
        28,  1, 39, 30,  1, 39, 50,  9,  1, 39])


## Linear algebra heads-up

Almost everything a transformer does is **matrix multiplication**: embeddings are
row look-ups in a matrix, linear layers are `x @ W + b`, attention is `softmax(QKᵀ)V`.
Make sure you are comfortable with shapes like `(B, T, C)` — batch, time, channels —
and with the rule `(m×n) @ (n×p) = (m×p)`. Notebook 02 has a hands-on refresher.

## Train and validation splits

80 % of the text is used for training and the last 20 % for validation. The model never
trains on validation data, so validation loss tells us whether we *generalize* or just
*memorize* (overfit).

In [5]:
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

185845 46462


## Premise of the bigram model

A **bigram** model predicts the next character using only the current character:
P(next | current). It is the simplest possible language model and gives us the full
training pipeline (data → model → loss → optimizer → generation) which we later
upgrade to a GPT.

## Inputs and targets

From a chunk of `block_size + 1` characters we get `block_size` training examples: for
each position `t`, the context `x[:t+1]` should predict `y[t] = x[t+1]`. Targets are the
inputs shifted by one.

In [6]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print('when input is', context.tolist(), 'target is', target.item())

when input is [28] target is 39
when input is [28, 39] target is 42
when input is [28, 39, 42] target is 39
when input is [28, 39, 42, 39] target is 44
when input is [28, 39, 42, 39, 44] target is 32
when input is [28, 39, 42, 39, 44, 32] target is 49
when input is [28, 39, 42, 39, 44, 32, 49] target is 1
when input is [28, 39, 42, 39, 44, 32, 49, 1] target is 25


## Batch size hyperparameter and `get_batch`

GPUs are parallel machines, so we process many independent chunks at once. `batch_size`
is how many sequences are stacked in one forward pass; `block_size` is the length of
each sequence. We sample random starting offsets and stack the slices.

## Switching from CPU to CUDA

Pick the fastest available device once and move every tensor/model to it with `.to(device)`.

In [7]:
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

batch_size = 4
torch.manual_seed(1337)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

x, y = get_batch('train')
print('inputs:', x.shape)
print(x)
print('targets:', y.shape)
print(y)

device: cpu
inputs: torch.Size([4, 8])
tensor([[73, 61, 58, 66,  1, 54, 65, 65],
        [ 1,  3, 54, 67, 57,  1, 33,  5],
        [ 1, 61, 58, 71,  1, 59, 71, 68],
        [58,  1, 66, 54, 67,  9,  1, 65]])
targets: torch.Size([4, 8])
tensor([[61, 58, 66,  1, 54, 65, 65,  1],
        [ 3, 54, 67, 57,  1, 33,  5, 66],
        [61, 58, 71,  1, 59, 71, 68, 66],
        [ 1, 66, 54, 67,  9,  1, 65, 68]])
